<a href="https://colab.research.google.com/github/OswaldoVelezDev/Trabajo-web-scraping/blob/main/Velez_Jose_Velez_Andres_Analizando_EA1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#instalacion
!pip -q install scrapy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 379.8/379.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.4/493.4 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.5/283.5 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.5/107.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 5.3 MB/s eta 0:00:00


In [ ]:
#creacion del spider

%%writefile spider_algolia.py
import json
from urllib.parse import urlencode, urljoin
import scrapy

APP_ID = "QX5IPS1B1Q"
API_KEY = "ODNiNzFhOTg5MWEyNWYxZGIyMmMyODFjYjNkMWU3NjVhNjUzYzgzZWUwMjkwODI3NmM4MDliNTY4OTEzODc5NCZmaWx0ZXJzPWF2YWlsYWJsZUJ1eUJ1dE5vdFZpc2libGVfYm9vbGVhbiUzQWZhbHNl"
INDICE = "alkostoIndexAlgoliaPRD"
ENDPOINT = f"https://{APP_ID.lower()}-dsn.algolia.net/1/indexes/{INDICE}/query"


def como_texto(v):
    if v is None:
        return None
    if isinstance(v, str):
        return v
    if isinstance(v, list):
        return como_texto(v[0]) if v else None
    if isinstance(v, dict):
        return como_texto(next(iter(v.values()), None))
    return str(v)


def como_lista(v):
    if v is None:
        return None
    if isinstance(v, list):
        return ", ".join(str(x) for x in v)
    return str(v)


class AlgoliaSpider(scrapy.Spider):
    name = "alkosto_algolia"

    custom_settings = {
        "USER_AGENT": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "ROBOTSTXT_OBEY": False,
        "HTTPERROR_ALLOW_ALL": True,
        "DOWNLOAD_DELAY": 1,
        "LOG_LEVEL": "INFO",
    }

    def __init__(self, filtros="allcategories_string_mv:BI_M009_ALKOS", paginas=3, inicio=0, hits=25, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.filtros = filtros
        self.paginas = int(paginas)
        self.inicio = int(inicio)
        self.hits = int(hits)
        self.vistos = set()
        self.logger.info(f"Spider versión 4 (Algolia) | filtros: {self.filtros}")

    def peticiones(self):
        for n in range(self.inicio, self.inicio + self.paginas):
            params = urlencode({"query": "", "filters": self.filtros, "hitsPerPage": self.hits, "page": n})
            yield scrapy.Request(
                ENDPOINT,
                method="POST",
                body=json.dumps({"params": params}),
                headers={
                    "X-Algolia-Application-Id": APP_ID,
                    "X-Algolia-API-Key": API_KEY,
                    "Content-Type": "application/json",
                    "Origin": "https://www.alkosto.com",
                    "Referer": "https://www.alkosto.com/",
                },
                callback=self.parse,
                cb_kwargs={"pagina": n},
                dont_filter=True,
            )

    async def start(self):
        for r in self.peticiones():
            yield r

    def start_requests(self):
        return self.peticiones()

    def parse(self, response, pagina):
        if response.status != 200:
            self.logger.error(f"Página {pagina}: HTTP {response.status} -> {response.text[:300]}")
            return

        datos = response.json()
        hits = datos.get("hits", [])
        self.logger.info(
            f"Página {pagina} | HTTP 200 | nbHits={datos.get('nbHits')} "
            f"nbPages={datos.get('nbPages')} | hits recibidos={len(hits)}"
        )

        for h in hits:
            sku = h.get("code_string") or h.get("objectID")
            if sku in self.vistos:
                continue
            self.vistos.add(sku)

            url = h.get("url_es_string")
            yield {
                "Pagina": pagina,
                "SKU": sku,
                "Nombre": h.get("name_text_es"),
                "Marca": como_texto(h.get("brand_string_mv")) or h.get("marca_text"),
                "Precio (COP)": h.get("pricevalue_cop_double"),
                "Precio con descuento": h.get("discountprice_double"),
                "Descuento": h.get("percentagediscount_string"),
                "Almacenamiento": como_lista(h.get("memoria-interna_string_mv")),
                "Color": como_lista(h.get("color_string_mv")),
                "Disponibilidad": h.get("stocklevelstatus_string"),
                "Stock": h.get("stocklevel_int"),
                "Visible": h.get("visible_boolean"),
                "URL": urljoin("https://www.alkosto.com", url) if url else None,
            }

Writing spider_algolia.py


In [ ]:
#ejecucion del spider
!rm -f productos_alkosto.json
!scrapy runspider spider_algolia.py -a paginas=3 -a inicio=0 -a hits=25 -O productos_alkosto.json

2026-10-19 22:50:57 [scrapy.utils.log] INFO: Scrapy 2.19.0 started (bot: scrapybot)
2026-10-19 22:50:57 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.1.3',
 'libxml2': '2.14.6',
 'cssselect': '1.6.0',
 'parsel': '1.12.1',
 'w3lib': '2.5.0',
 'Twisted': '26.4.0',
 'Python': '3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]',
 'pyOpenSSL': '26.4.0 (OpenSSL 4.0.3 29 Sep 2026)',
 'cryptography': '50.0.2',
 'Platform': 'Linux-6.6.122+-x86_64-with-glibc2.39'}
2026-10-19 22:50:57 [scrapy.crawler] DEBUG: Using AsyncCrawlerProcess
2026-10-19 22:50:57 [asyncio] DEBUG: Using selector: EpollSelector
2026-10-19 22:50:57 [alkosto_algolia] INFO: Spider versión 4 (Algolia) | filtros: allcategories_string_mv:BI_M009_ALKOS
2026-10-19 22:50:57 [scrapy.addons] INFO: Enabled addons:
[]
2026-10-19 22:50:57 [scrapy.extensions.telnet] INFO: Telnet Password: b89845226573ac8b
2026-10-19 22:50:57 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.logcou

In [ ]:
#Carga y verificación de los datos
import json
import pandas as pd

with open("productos_alkosto.json", encoding="utf-8") as f:
    df_productos = pd.DataFrame(json.load(f))

print(f"Total de productos obtenidos: {len(df_productos)}")
print(df_productos.groupby("Pagina").size())
print("\nDisponibilidad:")
print(df_productos["Disponibilidad"].value_counts(dropna=False))
print("\nVisible:")
print(df_productos["Visible"].value_counts(dropna=False))

display(df_productos)

Total de productos obtenidos: 59
Pagina
0    25
1    25
2     9
dtype: int64

Disponibilidad:
Disponibilidad
inStock    59
Name: count, dtype: int64

Visible:
Visible
true    59
Name: count, dtype: int64


,Pagina,SKU,Nombre,Marca,Precio (COP),Precio con descuento,Descuento,Almacenamiento,Color,Disponibilidad,Stock,Visible,URL
0,0,195951409382,iPhone 18 Pro Max 256GB 5G Negro,APPLE,6139010.0,NaN,None,256 GB,Negro,inStock,186.0,true,https://www.alkosto.com/iphone-18-pro-max-256g...
1,0,195950638998,iPhone 17 Pro Max 256GB 5G Naranja Cósmico,APPLE,6949010.0,5499010.0,-20%,256 GB,Naranja,inStock,1.0,true,https://www.alkosto.com/iphone17-promax-256gb-...
2,0,195951415055,iPhone 18 Pro 256GB 5G Azul Glaciar,APPLE,5569010.0,NaN,None,256 GB,Azul,inStock,6.0,true,https://www.alkosto.com/iphone-18-pro-256gb-5g...
3,0,195951415048,iPhone 18 Pro 256GB 5G Borgoña,APPLE,5569010.0,NaN,None,256 GB,None,inStock,37.0,true,https://www.alkosto.com/iphone-18-pro-256gb-5g...
4,0,195951415031,iPhone 18 Pro 256GB 5G Plata,APPLE,5569010.0,NaN,None,256 GB,Plateado,inStock,63.0,true,https://www.alkosto.com/iphone-18-pro-256gb-5g...
5,0,195951409450,iPhone 18 Pro Max 512GB 5G Azul Glaciar,APPLE,7049010.0,NaN,None,512 GB,Azul,inStock,108.0,true,https://www.alkosto.com/iphone-18-pro-max-512g...
6,0,195951409429,iPhone 18 Pro Max 512GB 5G Negro,APPLE,7049010.0,NaN,None,512 GB,Negro,inStock,31.0,true,https://www.alkosto.com/iphone-18-pro-max-512g...
7,0,195949035913,iPhone 15 128 GB 5G Negro,APPLE,4599010.0,3299010.0,-28%,128 GB,Negro,inStock,441.0,true,https://www.alkosto.com/iphone-15-128gb-5g-neg...
8,0,195951415024,iPhone 18 Pro 256GB 5G Negro,APPLE,5569010.0,NaN,None,256 GB,Negro,inStock,115.0,true,https://www.alkosto.com/iphone-18-pro-256gb-5g...
9,0,195949036453,iPhone 15 128 GB 5G Azul,APPLE,4599010.0,3299010.0,-28%,128 GB,Azul,inStock,339.0,true,https://www.alkosto.com/iphone-15-128gb-5g-azu...
